# Trial of Reflection

### Part 1 - Kevin: Refactored Loop

In [ ]:
import numpy as np


def relu(x):
    return np.maximum(0, x)


def relu_deriv(x):
    return (x > 0).astype(float)


def train(tells, strike, alpha, epochs, hidden_size, seed):
    """
    Trains a 2-layer neural network with ReLU hidden activation using SGD.
    Organized into four visual movements with shape annotations on every math line.
    """
    np.random.seed(seed)

    # Input and output dimensions based on input shapes
    input_size = tells.shape[1] if tells.ndim > 1 else len(tells[0])
    
    # Ensure goal array is 2D with shape (N, 1) or (N, output_dim)
    goals = np.array(strike).reshape(-1, 1)
    output_size = goals.shape[1]

    # Initialize weight matrices with uniform distribution in [-1, 1]
    weights_0_1 = 2 * np.random.random((input_size, hidden_size)) - 1  # (3, 4)
    weights_1_2 = 2 * np.random.random((hidden_size, output_size)) - 1  # (4, 1)

    error_history = []

    for epoch in range(epochs):
        total_error = 0.0

        for i in range(len(tells)):
            # Extract single observation with batch size of 1
            layer_0 = tells[i : i + 1]  # (1, 3)
            goal = goals[i : i + 1]  # (1, 1)

            # FORWARD
            layer_1 = relu(layer_0 @ weights_0_1)  # (1, 4)
            layer_2 = layer_1 @ weights_1_2  # (1, 1)

            # COMPARE
            error = np.sum((layer_2 - goal) ** 2) 
            total_error += error 

            # BACKWARD
            layer_2_delta = layer_2 - goal  # (1, 1)
            layer_1_delta = (layer_2_delta @ weights_1_2.T) * relu_deriv(layer_1)  # (1, 4)

            # UPDATE
            weights_1_2 -= alpha * (layer_1.T @ layer_2_delta)  # (4, 1)
            weights_0_1 -= alpha * (layer_0.T @ layer_1_delta)  # (3, 4)

        error_history.append(total_error)

    return error_history


if __name__ == "__main__":
    # Example Trial Dataset setup (matching Week 5)
    tells = np.array([[1, 0, 1], [0, 1, 1], [0, 0, 1], [1, 1, 1]])
    strike = np.array([1, 1, 0, 0])

    # Run refactored loop
    history = train(tells, strike, alpha=0.2, epochs=60, hidden_size=4, seed=1)
    print(f"Final epoch error: {history[-1]:.10f}")

: 

#### Explanation

This remade version of the week 5 model self-explains how the diagram is created simply through its structure
and makes the neural network much easier to follow along with simply by better design.

### Part 2 - Elijah: Architectures


#### Architectures

##### Wider

layer_0 --[ weights_0_1 (3, 32), relu ] --> layer_1 --[ weights_1_2 (32, 1) ] --> layer_2
 (1, 3)                                     (1, 32)                             (1,1)

Weights: 3x32 + 32x1 = 128

Rationale: This model would be excellent at capturing all of the possible combinations of interactions between the inputs and memorizing the data. This wastes a lot of compute, and due to having only one activation function, would struggle to capture high-dimensional relationships

##### Deeper

layer_0 --[ weights_0_1 (3,4), relu ] --> layer_1 --[ weights_1_2 (4, 4), relu ] --> layer_2 --[ weights_2_3 (4, 4), relu ] -->
(1, 3)                                      (1, 4)                                      (1,4)
layer_3 --[ weights_3_4 (4, 1) ] --> layer_4
(1, 4)                                (1, 1)

Weights: 3x4 + 4x4 + 4x4 + 4x1 = 48

Rationale: This model is excellent for capturing very high-dimensional relationships while also being very efficient with compute. Some issues can arise with this model due to the narrow hidden layer size because it may miss some specific combinations of inputs and that will cause issues for those combinations.

##### Multi-Output

layer_0 --[ weights_0_1 (3, 8), relu ] --> layer_1 --[ weights_1_2 (8, 4) ] --> layer_2
 (1, 3)                                     (1, 8)                             (1,4)

Weights: 3x8 + 8x4 = 56

Rationale: This model is good at capturing simpler, lower dimensional relationships between the inputs and outputs. It will struggle if any of the outputs require more complex computations, as the model cannot capture much of a higher dimensional relationship or simply more of the possible combinations of layer values.

##### Our Design

Given basic team statistics (yds per game, yds allowed, win percentage, avg game score) for both teams, predict final scores for both teams.

layer_0 --[ weights_0_1 (8, 32), relu ] --> layer_1 --[ weights_1_2 (32, 32), relu ] -->
(1, 8)                                      (1, 32)             
layer_2 --[ weights_2_3 (32, 2) ] --> layer_3
(1, 32)                                 (1, 2)

Weights: 8x32 + 32x32 + 32x2 = 1344

Rationale: This model is good at taking several inputs and evaluating many possible combinations of them, as well as capturing a decent amount of non-linearity. This model requires a decent amount of compute and may still struggle on very complex relationships.

#### Explanation

These models show the conceptualization of neural networks. By properly creating visual representations of networks, they make them much simpler to understand and code.

### Part 3 - Robert: Build From Diagram

In [ ]:
import numpy as np

# Relu function
def relu(x):
    return np.maximum(0, x)

# Deriv of Relu
def relu_deriv(y):
    return (y > 0).astype(float)

# Train from digram function to run through our layers to reach a (1, 1) matrix on layer 3
def train_from_diagram(tells, strike, alpha, epochs, seed):
    # seed for the following program
    np.random.seed(seed)
    
    # Diagram Shapes
    weights_0_1 = 2 * np.random.random((3, 8)) - 1
    weights_1_2 = 2 * np.random.random((8, 4)) - 1
    weights_2_3 = 2 * np.random.random((4, 1)) - 1

    error_history = []

    for epoch in range (epochs):

        for i in range(len(tells)):
            total_error = 0

            # running through the layers
            layer_0 = tells[i: i+1] # (1, 3)
            layer_1 = relu(layer_0 @ weights_0_1) # (1, 8)
            layer_2 = relu(layer_1 @ weights_1_2) # (1, 4)
            layer_3 = layer_2 @ weights_2_3 # (1, 1)

            # increase total error on each runthrough
            layer_3_delta = layer_3 - strike[i:i+1]
            total_error += np.sum(layer_3_delta ** 2)

            # Backpropogation (Find delta for each layer)
            layer_2_delta = (layer_3_delta @ weights_2_3.T) * relu_deriv(layer_2)
            layer_1_delta = (layer_2_delta @ weights_1_2.T) * relu_deriv(layer_1)

            # update weights
            weights_2_3 -= alpha * layer_2.T @ layer_3_delta
            weights_1_2 -= alpha * layer_1.T @ layer_2_delta
            weights_0_1 -= alpha * layer_0.T @ layer_1_delta

        # Update error history for this iteration
        error_history.append(total_error)

        # Print total error every 30 epochs
        if epoch % 30 == 0:
            print(f"Epoch {epoch}: Total Error = {total_error:.10f}")

    # Print final pred vs goals

    print("\n Final Predictions vs. Goals")
    for i in range(len(tells)):
        layer_0 = tells[i: i+1]
        layer_1 = relu(layer_0 @ weights_0_1)
        layer_2 = relu(layer_1 @ weights_1_2)
        layer_3 = layer_2 @ weights_2_3

        print(f"Prediction: {layer_3[0, 0]:.6f}, Goal: {strike[i]}")


    return error_history

if __name__ == "__main__":
    # Example Trial Dataset setup (matching Week 5)
    tells = np.array([[1, 0, 1], [0, 1, 1], [0, 0, 1], [1, 1, 1]])
    strike = np.array([1, 1, 0, 0])

    # Run refactored loop
    history = train_from_diagram(tells, strike, alpha=0.1, epochs=150, seed=4)
    print(f"Final epoch error: {history[-1]:.10f}")

""" Comment Block
weights_0_1 connects layer_0 to layer_1.
It must be (3, 8) because layer_0 has 3 values
and layer_1 has 8 values.
 
weights_1_2 connects layer_1 to layer_2.
It must be (8, 4) because layer_1 has 8 values
and layer_2 has 4 values.
 
weights_2_3 connects layer_2 to layer_3.
It must be (4, 1) because layer_2 has 4 values
and layer_3 has 1 value.
"""


#### Explanation

This showcases our ability to go from the visualization to model, and the flow of the code that self explains this transition.

### Part 4 - Brady: Unit Tests

In [ ]:
import numpy as np

from part1_refactored_loop import train
from part3_build_from_diagram import train_from_diagram

## Test 1
def test_refactor_correctness():
  tells = np.array([
    [1, 0, 1],
    [0, 1, 1],
    [0, 0, 1],
    [1, 1, 1]
  ])
  
  strike = np.array([1, 1, 0, 0])
  
  history = train(
    tells, strike, alpha=0.2, epochs=60, hidden_size=4, seed=1)
  
  expected = 0.0000150556
  
  np.testing.assert_allclose(
    history[-1], expected, atol=1e-9)

## Test 2
def test_shape_sanity():
  tells = np.array([
    [1, 0, 1],
    [0, 1, 1],
    [0, 0, 1],
    [1, 1, 1]
])
  
  strike = np.array([1, 1, 0, 0])
  
  hidden_size = 4
  
  np.random.seed(1)
  
  weights_0_1 = 2 * np.random.random((3, hidden_size)) - 1
  weights_1_2 = 2 * np.random.random((hidden_size, 1)) - 1
  
  layer_0 = tells[0:1]
  goal = strike[0:1].reshape(1, 1)
  
  layer_1 = np.maximum(0, layer_0 @ weights_0_1)
  layer_2 = layer_1 @ weights_1_2
  
  layer_2_delta = layer_2 - goal
  layer_1_delta = (
  layer_2_delta @ weights_1_2.T
  ) * (layer_1 > 0).astype(float)
  
  assert layer_0.shape == (1, 3)
  assert layer_1.shape == (1, 4)
  assert layer_2.shape == (1, 1)
  
  assert layer_1_delta.shape == (1, 4)
  assert layer_2_delta.shape == (1, 1)
  assert weights_0_1.shape == (3, 4)
  assert weights_1_2.shape == (4, 1)

## Test 3
def test_deeper_net_runs():
  tells = np.array([
    [1, 0, 1],
    [0, 1, 1],
    [0, 0, 1],
    [1, 1, 1]
])
  
  strike = np.array([1, 1, 0, 0])
  
  epochs = 10
  
  history = train_from_diagram( tells, strike, alpha=0.1, epochs=epochs, seed=4)
  
  assert len(history) == epochs

## Test 4
def test_deeper_net_converges():
 tells = np.array([
    [1, 0, 1],
    [0, 1, 1],
    [0, 0, 1],
    [1, 1, 1]
 ])
  
 strike = np.array([1, 1, 0, 0])
  
 alpha = 0.1
 epochs = 150
 seed = 4
  
 history = train_from_diagram(tells, strike, alpha=alpha, epochs=epochs, seed=seed)
  
 assert history[-1] < 1e-3

## Test 5
def test_determinism():
  tells = np.array([
    [1, 0, 1],
    [0, 1, 1],
    [0, 0, 1],
    [1, 1, 1]
])
  
  strike = np.array([1, 1, 0, 0])
  
  history1 = train_from_diagram(tells, strike, alpha=0.1, epochs=150, seed=4)
  
  history2 = train_from_diagram(tells, strike, alpha=0.1, epochs=150, seed=4)
  
  np.testing.assert_allclose(history1, history2)

## Main
if __name__ == "__main__":
  tests = [
    value
    for name, value in globals().items()
    if name.startswith("test_") and callable(value)
  ]
  for test in tests:
    try: 
      test()
      print(f"PASS: {test.__name__}")
    except AssertionError as e:
      print(f"FAIL: {test.__name__}")
      print(e)


#### Explanation

Unit tests are always importance because they prove we know what we're doing.

### Part 5 - Colson: Skill Importance

The diagram discipline might become the most important skill we learn this semester because diagrams set 
up the framework for what we will be working on. If I see a diagram of a model, I should be able to figure 
out what type of code I will need just by looking at the model. This is definitely not an easy skill but 
it is one that is necessary. For me, it was difficult to write the network I have never seen before. I 
struggled a lot and took to referring to other models from our notes to fill in gaps I had in my logic. 
It was far from simple but by looking back, I did have reference points in my previous code to figure 
things out. I think what Korrith's disciples lost was the ability to see the full picture of what they 
are building. Without the diagram, you don't necessarily know where you are going to start and stop in a 
given program unless it is explicitly told to you and gien time you may never get that.

#### Explanation

This is a great summary of why we are doing any of this at all.